# Getting Started with Automated-LLM-Probes

In [5]:
from __future__ import annotations
import os,time,tqdm,pandas as pd
from pathlib import Path
import automated_intelligence_tests as ait
import automated_llm_probes as alp
assert os.environ.get("OPENAI_API_KEY")

### Models available

In [2]:
models = alp.ready_models()
print(f"|Models available: {len(models)}|\n")
for m in models:
    print(f"  {m['name'][:20]:20s} {m['vendor'][:11]:12s} \
{m['api'][:11]:12s} {m['model_id'][:12]:14s} {m['status'][:12]:12s}")

|Models available: 78|

  grok-code-fast       xai          spacexai     grok-code-fa   ok          
  grok-4.20-non-reason xai          spacexai     grok-4.20-03   ok          
  grok-4.20-reasoning  xai          spacexai     grok-4.20-03   ok          
  grok-4.3             xai          spacexai     grok-4.3       ok          
  grok-4.5             xai          spacexai     grok-4.5       ok          
  grok-build-0.1       xai          spacexai     grok-build-0   ok          
  grok-4.6             xai          spacexai     grok-4.6       ok          
  grok-4.20-multi-agen xai          spacexai     grok-4.20-mu   unverified;d
  hunyuan-3            tencent      hunyuan      hy3            ok          
  hunyuan-lite         tencent      hunyuan      hunyuan-lite   unverified;d
  qwen-turbo           qwen         qwen         qwen-turbo     ok          
  qwen-3-235b-instruct qwen         qwen         qwen3-235b-a   ok          
  qwen-max             qwen         qwen         qwe

### Probe models

In [6]:
def probe_models(model_rows,prompt="Reply with: ok"):
    retries = alp.MAX_RETRIES
    alp.MAX_RETRIES = 1
    try:
        for model in tqdm.tqdm(model_rows):
            start = time.time()
            try:
                model['reply'] = alp.call_model(
                    model,[{"role": "user", "content": prompt}])
                model["status"] = "ok" if model['reply'].strip() else "failed"
                model["errors"] = None
            except Exception as e:
                model["errors"] = str(e).lower()
            model["speed_per_call"] = int((time.time() - start) * 1000)
    finally:
        alp.MAX_RETRIES = retries
    return model_rows

models = alp.load_models()
probes = probe_models(models)
probes = pd.DataFrame(probes).set_index('name')

100%|████████████████████████████████████████████████████████████████████| 83/83 [02:19<00:00,  1.68s/it]


In [8]:
probes.to_csv('./models.csv')
probes.head()

,vendor,api,model_id,release_date,temperature,speed_per_call,status,reply,errors
name,,,,,,,,,
grok-code-fast,xai,spacexai,grok-code-fast,26-Aug-25,0.5,2117,ok,ok,None
grok-4.20-non-reasoning,xai,spacexai,grok-4.20-0309-non-reasoning,9-Mar-26,0.5,473,ok,ok,None
grok-4.20-reasoning,xai,spacexai,grok-4.20-0309-reasoning,9-Mar-26,0.5,1564,ok,ok,None
grok-4.3,xai,spacexai,grok-4.3,30-Apr-26,0.5,1356,ok,ok,None
grok-4.5,xai,spacexai,grok-4.5,8-Jul-26,0.5,997,ok,ok,None


### Probed tasks

In [ ]:
path = Path('./data/'); n=1
print(f"|Probed tasks: {len([p for p in path.iterdir() if p.is_dir()])}|\n")
for d in path.iterdir():
    if (d.is_dir()) and ('.' not in d.name):
        count = sum(1 for f in d.rglob('*') if f.is_file())
        print(f"  {n}. {d.name.upper()[:7]:8}:  {count}"); n+=1

### Tests availabe

In [4]:
ait_counts = ait.list_available_tests()
print(f'|Available tests: {len(ait_counts)}|\n')
for i,v in ait_counts.items():
    print(f'  {i} : {v}')

|Available tests: 4|

  AUT : Alternative Uses Task
  CAT : Convergent Association Task
  CWT : Creative Writing Task
  DAT : Divergent Association Task


### Trial-run

In [5]:
models_to_try = [
    'Claude Haiku 4.5',
    'Claude Opus 4.5',
    'Claude Opus 4.7',
    'Claude Opus 5',
    'Claude Sonnet 4.5',
    'GPT-3.5-Turbo',
    'GPT-4-Turbo',
    'GPT-4o',
    'GPT-4o-mini',
    'GPT-5.4',
    'Grok 4.2',
    'Grok 4.3',
    'Grok 4.5',
    'Grok Build 0.1',
    'Llama-3.1 8b',
    'Llama-3.2 3b',
    'Llama-4-Maverick',
    'Llama-4-Scout',
    "Llama-4-Guard-12b"]

cues_to_try = [
    'brick','paperclip']

models_to_try = [m for m in models if m["name"] in models_to_try]
sorted([m['name'] for m in models_to_try])

['Claude Haiku 4.5',
 'Claude Opus 4.5',
 'Claude Opus 4.7',
 'Claude Opus 5',
 'Claude Sonnet 4.5',
 'GPT-3.5-Turbo',
 'GPT-4-Turbo',
 'GPT-4o',
 'GPT-4o-mini',
 'GPT-5.4',
 'Grok 4.2',
 'Grok 4.3',
 'Grok 4.5',
 'Grok Build 0.1',
 'Llama-3.1 8b',
 'Llama-3.2 3b',
 'Llama-4 Maverick',
 'Llama-4 Scout']

In [6]:
alp.collect("AUT",models=models_to_try,n_per_model=n,cue=cues_to_try)

  Grok 4.2: 620/4 done — skip
  Grok 4.3: 620/4 done — skip
  Grok 4.5: 620/4 done — skip
  Grok Build 0.1: 620/4 done — skip
  GPT-3.5-Turbo: 731/4 done — skip
  GPT-4o: 681/4 done — skip
  GPT-5.4: 618/4 done — skip
  GPT-4o-mini: 620/4 done — skip
  GPT-4-Turbo: 620/4 done — skip
  Llama-4 Scout: 620/4 done — skip
  Llama-4 Maverick: 595/4 done — skip
  Llama-3.2 3b: 590/4 done — skip
  Llama-3.1 8b: 590/4 done — skip
  Claude Sonnet 4.5: 627/4 done — skip
  Claude Haiku 4.5: 620/4 done — skip
  Claude Opus 4.5: 620/4 done — skip
  Claude Opus 4.7: 376/4 done — skip
  Claude Opus 5: 350/4 done — skip


### Load functions

In [7]:
from __future__ import annotations
import re, pandas as pd
import glove_word_embeddings as gwe

def parse_dat(raw):
    text = str(raw or "").strip().strip('"').strip("'")
    tokens = re.split(r"[,\n\r]+", text)
    nouns = [n for n in (gwe.pre.clean_word(t) for t in tokens) if n][:10]
    return nouns + [""] * (10 - len(nouns))

def parse_aut(raw):
    uses = []
    text = re.sub(r"<br\s*/?>", "\n", str(raw or ""), flags=re.I)
    for line in re.split(r"[\n\r,;]+", text):
        line = re.sub(r"^\s*[\d\.\)\-]+\s*", "", line)
        if toks := [t for t in (gwe.pre.clean_word(t) for t in line.split()) if t]:
            uses.append(" ".join(toks))
    return ", ".join(uses)

def parse_cwt(raw):
    text = re.sub(r"^#+\s*.*$", "", str(raw or ""), flags=re.M)
    text = re.sub(r"^\s*Title:.*$", "", text, flags=re.M | re.I)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def load_task(df,task) -> pd.DataFrame:
    if task == "dat":
        parsed = df["raw"].map(parse_dat)
        df[[f"noun_{i}" for i in range(10)]] = parsed.tolist()
        df["response_clean"] = parsed.map(lambda x: ", ".join(n for n in x if n))
        extra = [f"noun_{i}" for i in range(10)]
    elif task == "aut":
        df["object"] = df["prompt"].str.extract(
            r"object: (.+?)\?", expand=False).str.strip()
        df["response_clean"] = df["raw"].map(parse_aut)
        extra = ["object"]
    elif task == "cwt":
        parts = (
            df["prompt"].str.extract(r"words?(?:\(s\))?:\s*(.+?)\.", expand=False)
            .fillna("")
            .map(lambda s: [w.strip() for w in str(s).split(",") if w.strip()])
        )
        parts = parts.map(lambda xs: ["".join(xs)] if xs and max(map(len, xs)) == 1 else xs)
        cue_df = pd.DataFrame(parts.tolist(), index=df.index).reindex(columns=range(3))
        df[["cue_0", "cue_1", "cue_2"]] = cue_df.to_numpy()
        df["response_clean"] = df["raw"].map(parse_cwt)
        extra = ["cue_0", "cue_1", "cue_2"]
    else:
        raise ValueError(f"Unknown task: {task}")
    cols = ["task", "model_name", "model_id", 
            "provider", "rep", "temperature_std"] + extra + [
        "prompt", "response_clean", "ts_utc", "hash"]
    return df[[c for c in cols if c in df.columns]].sort_values(
        ["model_name", "rep"]).reset_index(drop=True)
        
print('All functions loaded...')

All functions loaded...


### Parse & merge data

In [10]:
for task in (
    "dat", 
    "aut",
    "cwt"
):
    print(f"Parsing {task.upper()}...")
    task = task.lower()
    df = pd.DataFrame.from_dict(alp.parse_and_merge(task),orient='index')
    df = load_task(df,task)
    print(df.shape)
    df.to_csv(f"./data/{task.upper()}_AI_2026.csv", index=False)

Parsing DAT...


dat: 100%|██████████████████████████████████████████████████████████| 8428/8428 [00:30<00:00, 274.05it/s]


(8293, 19)
Parsing AUT...


aut: 100%|████████████████████████████████████████████████████████| 12588/12588 [00:48<00:00, 257.24it/s]


(12566, 10)
Parsing CWT...


cwt: 100%|████████████████████████████████████████████████████████| 14353/14353 [01:04<00:00, 223.42it/s]


(14246, 12)


In [9]:
m = "Claude Opus 5"   # pickle name; also try "claude-opus-5"
raw = pd.DataFrame.from_dict(alp.parse_and_merge("dat"), orient="index")
print(raw.model_name.value_counts())
print(raw.loc[raw.model_name.eq(m), "temperature_std"].fillna("default").value_counts())
print("empty raw", raw.loc[raw.model_name.eq(m), "raw"].eq("").sum() if "raw" in raw.columns else "n/a")

dat: 100%|██████████████████████████████████████████████████████████| 8428/8428 [00:10<00:00, 767.91it/s]

model_name
Claude Opus 4.7      650
Claude Opus 4.5      400
Claude Sonnet 4.5    400
Claude Opus 5        400
Grok 4.3             400
Grok Build 0.1       400
Grok 4.5             400
Grok 4.2             400
Claude Haiku 4.5     400
Llama-3.1 8b         350
Llama-4 Scout        350
GPT-4-Turbo          350
GPT-4o-mini          350
Grok 4.6             350
GPT-3.5-Turbo        350
Llama-3.2 3b         350
Llama-4 Maverick     350
GPT-4o               350
GPT-5.6-Sol          350
GPT-5.4              350
DeepSeek-3.2         275
DeepSeek-Chat        275
Kimi-k2               20
DeepSeek-R1            5
Grok Code Fast         5
GPT-5-mini             5
GPT-5                  5
Qwen-Turbo             3
Name: count, dtype: int64
temperature_std
0.5        250
default    150
Name: count, dtype: int64
empty raw 0
